# From past likes to an explainable shortlist

Category cosine analysis · 17 October 2026

## 1. Start with the decision

A visitor chooses Paris. We want up to ten new places to consider, with a reason for each. Category matching is useful because we have attraction labels and earlier ratings, while review text and reliable exact coordinates are missing. It does not require model training.

Inputs: private prepared files in `assets/baseline_data`. Outputs: a new `runs/*_cosine_*` folder. The preparation receipt links these arrays to the source CSVs. This notebook analyses category cosine only; the other methods remain unchanged. No ranking-quality protocol or business-impact claim is approved by this run.

| Section | Input | Output |
|---|---|---|
| 2–3 | Small labelled illustration | Hand-checkable scores |
| 4–6 | Prepared review events and catalogue | Cutoff history, candidates and a real profile |
| 7–10 | Profile and candidate categories | Contributions, ties, lists and sensitivity |
| 11–13 | Controlled inputs and saved scores | Fallback cases, checks and measurement examples |

Historical categories are a single snapshot. Publication dates determine what review history we use, not exact visit dates.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, hashlib, uuid, os
import numpy as np
import pandas as pd
from IPython.display import display


# Find the Tourism folder by looking for its prepared data files.
LAB = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'assets/baseline_data').exists()))
DATA = LAB / 'assets/baseline_data'


# Give this run its own folder so earlier results are not overwritten.
OUT = LAB / 'Current/interactive/runs' / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_cosine_' + uuid.uuid4().hex[:6])
OUT.mkdir(parents=True)


# Choose the history cutoff, destination, minimum reviewer count and list length here.
CONFIG = dict(train_before='2018-01-01', city='France/Paris', min_item_positive_users=5, top_k=10)


# Save the settings beside the results so we know how this list was made.
(OUT / 'configuration.json').write_text(json.dumps(CONFIG, indent=2))
print('Output:', OUT)
print(
    'Candidate support 5 is retained from the earlier experiment; it is adjustable, not a proven optimum.',
)


### Read the result

Read the configuration before interpreting any result. This run produces historical category matching, not a current operating-hours or navigation service.

## 2. Give each place a small set of labels

Illustration only: use three categories so we can see every number. A 1 means the catalogue assigns the label; a 0 means it does not. A zero is not a user dislike. A place can have more than one label. The real calculation later uses all 20 categories.

In [ ]:
# Use invented places with three category columns so every input number is visible. These are not real reviews.
toy = pd.DataFrame(
    [[1, 0, 0], [1, 1, 0], [0, 0, 1]],
    index=['Museum A', 'Museum and park B', 'Shopping C'],
    columns=['Museum', 'Park', 'Shopping'],
)
display(toy)


# Normalize each place before averaging: a place with more labels must not get extra vector length.
toy_unit = toy.to_numpy(dtype=float) / np.linalg.norm(toy.to_numpy(dtype=float), axis=1, keepdims=True)


# This example visitor likes the first two places. Average those two normalized rows.
toy_average = toy_unit[:2].mean(axis=0)


# Normalize the average once more to obtain the visitor’s unit-length interest vector.
toy_profile = toy_average / np.linalg.norm(toy_average)


# Show both stages of the profile so we can see what normalization changed.
display(
    pd.DataFrame(
        {'average_before_final_normalization': toy_average, 'normalized_interest': toy_profile},
        index=toy.columns,
    ),
)


### Read the result

The two-label place contributes approximately 0.707 in each active category after normalization. The final visitor profile puts approximately 0.924 on museum and 0.383 on park. These are directions, not percentages that must sum to 100%.

## 3. What cosine measures

Our visitor liked Museum A and Museum and park B. We give each distinct liked place one contribution, then average. A vector is just an ordered list of numbers. Its length is the square root of the sum of squared entries. Dividing by that length keeps its direction but makes its length one.

Cosine = sum of matching entry products / (length of visitor vector × length of attraction vector). After both vectors have length one, we only need the sum of matching entry products. It measures category alignment. With these nonnegative features it lies between zero and one. It is not a chance of liking or visiting.

Pause before running: which should match better, another museum or a shopping-only place?

In [ ]:
# Create four possible category combinations for the hand calculation.
toy_candidates = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0]], dtype=float)


# Give each candidate a unit-length vector, just like the liked places.
toy_candidates /= np.linalg.norm(toy_candidates, axis=1, keepdims=True)


# One dot product per candidate gives cosine because both sides already have length one.
toy_scores = toy_candidates @ toy_profile


# Attach readable candidate names before displaying the scores.
toy_result = pd.DataFrame(
    {
        'candidate': ['Museum only', 'Park only', 'Shopping only', 'Museum and park'],
        'cosine': toy_scores,
    },
)
display(toy_result)


# Compare the calculated scores with the values we can derive by hand.
assert np.allclose(toy_scores, [np.cos(np.pi / 8), np.sin(np.pi / 8), 0, np.cos(np.pi / 8)])
print(
    'Museum-only and museum-and-park tie here. Cosine does not uniquely decide the final order.',
)


# Save the invented inputs and scores separately from the real-data results.
toy.to_csv(OUT / 'illustration_labels.csv')
toy_result.to_csv(OUT / 'illustration_scores.csv', index=False)


### Read the result

Museum-only and museum-and-park both score approximately 0.924; park-only scores 0.383 and shopping-only scores 0. A tie is possible even with different label combinations. The example separates absent category overlap from dislike.

## 4. Reconstruct the permitted history

Check input hashes, then keep one earliest valid rating per person and place. Conflicting valid ratings on that earliest day are excluded from preference labels. Ratings 4 and 5 are likes with equal weight. Every earlier review, even a low or invalid rating, removes a place from repeat suggestions. These rules describe review records, not complete visits.

In [ ]:
# Read a file in small pieces and calculate its fingerprint, without loading the whole file into memory.
def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


# Check that these prepared files are the same ones listed in the preparation record.
receipt = json.loads((DATA / 'preparation_receipt.json').read_text())
actual = {name: sha(DATA / name) for name in ['items.csv', 'categories.npy', 'users.npy', 'events.npy']}
assert actual == receipt['sha256_derived']
(OUT / 'input_hashes.json').write_text(json.dumps(actual, indent=2))


# Load attraction details, category labels and review records. Read the large review array from disk as needed.
items = pd.read_csv(DATA / 'items.csv', keep_default_na=False)
categories = np.load(DATA / 'categories.npy').astype(float)
events = np.load(DATA / 'events.npy', mmap_mode='r')


# Keep the row counts and category names together: array row numbers must refer to the same people and places.
NITEM = len(items)
NUSER = len(np.load(DATA / 'users.npy', mmap_mode='r'))
category_names = list(items.columns[4:])


# Stop if the attraction rows or the 20 category columns do not line up, or an attraction ID appears twice.
assert categories.shape == (NITEM, 20) and len(category_names) == 20 and items.attractionID.is_unique
assert np.isin(categories, [0, 1]).all()


# Make every nonempty attraction vector length one. Leave an empty category vector at zero.
unit_categories = categories / np.maximum(np.linalg.norm(categories, axis=1, keepdims=True), 1)


# Convert the cutoff to the same day-number format used in the review file.
cut = int(np.datetime64(CONFIG['train_before'], 'D').astype(np.int64))
prior = events['day'] < cut


# Remember every earlier reviewed place, even when its rating is low or invalid, so we do not suggest it again.
seen_pairs = np.unique(events['user'][prior].astype(np.int64) * NITEM + events['item'][prior])


# Only ratings 1 through 5 can describe a preference. A stored zero is not a dislike.
valid = np.flatnonzero((events['rating'] >= 1) & (events['rating'] <= 5))


# Give each person–place combination one numeric key: person row × number of attractions + attraction row.
pair = events['user'][valid].astype(np.int64) * NITEM + events['item'][valid]


# Put each person–place group together, with its earliest date first; rating breaks same-day ties.
order = np.lexsort((events['rating'][valid], events['day'][valid], pair))
ordered = valid[order]
keys = pair[order]


# Find where each new person–place group begins, then take its earliest valid review.
starts = np.r_[0, np.flatnonzero(np.diff(keys)) + 1]
first = ordered[starts]


# Check all ratings on that earliest day, not just the first row. Ignore later-day ratings for this conflict check.
same_day = events['day'][ordered] == np.repeat(events['day'][first], np.diff(np.r_[starts, len(keys)]))
ratings = events['rating'][ordered]


# If the smallest and largest first-day ratings differ, leave this pair out instead of choosing one rating.
conflict = np.minimum.reduceat(np.where(same_day, ratings, 127), starts) != np.maximum.reduceat(np.where(same_day, ratings, -1), starts)


# Keep one non-conflicting earliest rating per person and place, then keep only records before the cutoff.
dedup = np.array(events[first[~conflict]])
train = dedup[dedup['day'] < cut]
positive = train[train['rating'] >= 4]


# Count different people who gave each place 4 or 5. Deduplication above makes each person count once.
support = np.bincount(positive['item'], minlength=NITEM)


# For category interests, a liked place also needs a category label. CF can still use likes without labels.
usable = positive[categories[positive['item']].sum(axis=1) > 0]
history_count = np.bincount(usable['user'], minlength=NUSER)


# Keep the record counts and people count separate when showing what survived preparation.
prep = {
    'prepared_event_rows': len(events),
    'training_valid_pairs': len(train),
    'training_positive_pairs': len(positive),
    'training_usable_category_positive_pairs': len(usable),
    'all_period_conflicting_pairs': int(conflict.sum()),
    'people_with_usable_profile': int((history_count > 0).sum()),
}
display(pd.Series(prep).to_frame('count'))
(OUT / 'preparation_counts.json').write_text(json.dumps(prep, indent=2))


# Free the large temporary sorting arrays; keep the history and candidate inputs needed below.
del valid, pair, order, ordered, keys, starts, first, same_day, ratings


### Read the result

The table counts different things, so its rows should not be read as one continuous subtraction:

- **12,662,370 review records** are in the prepared file. They cover 2010–2019, including records after the date used to build this interest profile.
- **8,548,473 person–place records** remain before January 2018 after keeping one valid rating per person and place and removing first-day rating conflicts. For example, three reviews by the same person of the same museum contribute at most one record here.
- **7,561,728 of those records have a rating of 4 or 5.** These are the places we count as liked.
- **7,546,530 liked-place records also have a category label.** We can use these to build category interests. A liked place without a category cannot tell this calculation whether the person likes museums, parks or something else.
- **3,172,626 people have at least one such usable liked place.** We can calculate an interest profile for each of them. This counts people; the earlier rows count records.
- **3,181 person–place pairs have conflicting ratings on their earliest valid review day.** This count covers the full prepared period, 2010–2019. We leave those pairs out of preference labels instead of picking one of the conflicting ratings.

**What this means for recommendations:** one liked place with a category is enough to calculate a profile. It gives us a starting point, but one record tells us little about the person's wider interests. Whether the resulting recommendations are useful needs a separate check.

## 5. Choose the city before ranking

Suppose the visitor says, “I want to explore Paris.” We first remove places outside Paris. Only then do we compare the remaining places with the visitor's interests.

For this run, a Paris place must also meet two conditions:

1. **At least five different people gave it a rating of 4 or 5 before January 2018.** Five reviews from one person do not count as five people. We keep this setting from the earlier experiment so we can inspect it; five has not been shown to be the best number.
2. **It has at least one category label**, such as museum or park. Without a label, this category-based calculation has nothing to compare with the visitor's interests.

The first condition means some places will never reach the scoring step, even if they might suit the visitor. A new place with only two high-rating reviewers, for example, is excluded. The next table shows how much each condition reduces the list. We need to see that loss before deciding whether to keep the setting.

In [ ]:
# Keep only places in the visitor’s chosen city; city is not a cosine feature.
city = items.locationAttraction.eq(CONFIG['city']).to_numpy()
has_category = categories.sum(axis=1) > 0


# A place enters this list only if it is in the city, has enough different high-rating reviewers, and has a category.
candidate_mask = city & (support >= CONFIG['min_item_positive_users']) & has_category
candidates = np.flatnonzero(candidate_mask)


# Show how many places each rule removes. These count labels describe the current five-reviewer example.
audit = pd.DataFrame(
    {
        'stage': ['Exact city', 'City + >=5 positive reviewers', 'City + >=5 + category'],
        'attractions': [int(city.sum()), int((city & (support >= 5)).sum()), len(candidates)],
    },
)
display(audit)


# Show which category labels are available among the city’s attractions.
display(
    pd.DataFrame(
        {
            'feature': category_names,
            'city_attractions_with_label': categories[city].sum(axis=0).astype(int),
        },
    ),
)


# Save the filtering counts so the candidate choice can be checked later.
audit.to_csv(OUT / 'candidate_filters.csv', index=False)
print(
    'Features: category labels only. Ratings select likes; city filters; popularity breaks ties and fills gaps.',
)


### Read the result

**We start with 7,977 Paris places in the catalogue.**

After requiring at least five different people to have given a place 4 or 5 stars before January 2018, **1,765 places remain**. That requirement leaves out **6,212 places**.

All 1,765 remaining places already have at least one category label, so the category requirement removes **zero additional places**.

**Why this matters:** we are choosing recommendations from 1,765 places, not from all 7,977 Paris places. The 6,212 excluded places are not necessarily bad. They do not have enough recorded high-rating reviewers under our current rule. A new or less-reviewed place could be excluded even if it fits the visitor's interests.

The next decision is whether requiring five people gives us a useful starting list or removes too many worthwhile options. This table shows the size of the reduction; it does not prove that five is the right setting.

## 6. Follow one real person into a profile

Choose the first compact user index with at least three usable earlier likes including a Paris candidate. This deterministic example uses no later outcomes and is not a representative sample. One usable like can run the method; three is used only to give this walkthrough several records to inspect. All usable earlier likes, including other cities, contribute. Display the full history before interpreting it.

In [ ]:
# Find people with a usable earlier like at this destination, without looking at later reviews.
local_users = np.unique(usable['user'][candidate_mask[usable['item']]])


# Pick a fixed person with at least three likes so we can inspect several records. One like is enough to run the method.
DEMO_USER = int(local_users[history_count[local_users] >= 3][0])


# Get this person’s distinct liked places, including places in other cities.
liked = usable['item'][usable['user'] == DEMO_USER]


# Average their unit-length attraction vectors, then make the average length one for cosine scoring.
raw_profile = unit_categories[liked].mean(axis=0)
profile = raw_profile / np.linalg.norm(raw_profile)


# Show the actual liked places and supplied labels so we can explain where the interest weights came from.
history = items.iloc[liked][['attractionID', 'nameAttraction', 'locationAttraction']].copy()
history['category_labels'] = ['; '.join(np.array(category_names)[categories[i] > 0]) for i in liked]
history['rating'] = usable['rating'][usable['user'] == DEMO_USER]
display(history)


# Show the mean before final normalization and the finished weight for each category.
profile_table = pd.DataFrame(
    {
        'category': category_names,
        'average_normalized_place_weight': raw_profile,
        'final_interest_weight': profile,
    },
).sort_values(
    'final_interest_weight',
    ascending=False,
)
display(profile_table[profile_table.final_interest_weight > 0])
print('Anonymous compact user index:', DEMO_USER, '; distinct usable liked places:', len(liked))


# Save both the history and the profile so the explanation can be traced back to records.
history.to_csv(OUT / 'real_history.csv', index=False)
profile_table.to_csv(OUT / 'real_profile.csv', index=False)


### Read the result

Saved-run finding: the example has three usable likes—Le Manoir de Paris, Cathédrale Notre-Dame de Paris and Moulin Rouge. Shows have weight 0.813, landmarks 0.476, and fun/games 0.337 after normalization. Zero museum weight means no museum-labelled positive evidence in this profile; it is not an inferred dislike.

## 7. Explain a real score one category at a time

Each category contributes visitor weight × candidate weight. Add the contributions to recover the score. A high score means the labels line up with this summary of the recorded likes. It does not tell us whether the person liked a museum for its cafe, architecture or children’s activities. Those reasons are missing.

In [ ]:
# Score each destination candidate against this person’s normalized category interests.
scores = unit_categories[candidates] @ profile


# Mark the candidates this person already reviewed so they can be excluded.
seen = np.isin(DEMO_USER * NITEM + candidates, seen_pairs)
ids = items.attractionID.to_numpy()[candidates]


# Sort by score first, then historical high-rating reviewer count, then ID. np.lexsort reads the last key first.
order = np.lexsort((ids, -support[candidates], -scores))


# Only unseen places with positive category overlap count as interest matches.
positive_order = order[~seen[order] & (scores[order] > 0)]
assert len(positive_order) > 0


# Take the first remaining match so we can explain a real score, not just show a ranked list.
best = int(positive_order[0])
best_item = int(candidates[best])


# Multiply each user-category weight by the matching attraction weight; these products must add up to the score.
contribution = profile * unit_categories[best_item]


# Put the inputs and products side by side so we can see which categories caused this match.
explanation = pd.DataFrame(
    {
        'category': category_names,
        'visitor_weight': profile,
        'place_weight': unit_categories[best_item],
        'contribution': contribution,
    },
)
display(items.iloc[[best_item]][['attractionID', 'nameAttraction', 'locationAttraction']])
display(explanation[explanation.contribution > 0].sort_values('contribution', ascending=False))


# Check that the displayed category contributions recover the original candidate score.
assert np.isclose(contribution.sum(), scores[best])
print('Sum of contributions:', contribution.sum())
explanation.to_csv(OUT / 'top_item_contributions.csv', index=False)


### Read the result

Saved-run finding: Palais Garnier scores 0.911095 from show and landmark overlap. The table exposes the exact arithmetic. Broad labels do not establish access, opening hours or the reason this visitor enjoyed previous places.

## 8. A score does not settle every rank

Sort positive matches by cosine, then historical positive-reviewer count, then attraction ID. Popularity is a chosen tie rule, not part of the cosine number. Compare the old ID-only tie rule to make that distinction visible. Many attractions can share the same broad label pattern; no amount of decimal printing gives those identical vectors extra information.

In [ ]:
# Rebuild the older ordering that uses attraction ID alone to break score ties.
id_order = np.lexsort((ids, -scores))


# Rebuild the older ordering that uses attraction ID alone to break score ties.
id_order = id_order[~seen[id_order] & (scores[id_order] > 0)]
K = CONFIG['top_k']


# Compare the two rank orders; any difference here comes from the tie rule, not a new cosine score.
tie_check = pd.DataFrame(
    {
        'rank': np.arange(1, min(K, len(positive_order), len(id_order)) + 1),
        'popularity_tie_item': ids[positive_order[:K]],
        'id_tie_item': ids[id_order[:K]],
    },
)
display(tie_check)


# Count different category rows. Identical category rows cannot be separated by this representation.
unique_patterns = len(np.unique(categories[candidates], axis=0))


# Report how many candidates match and how many tie at the highest score.
geometry = {
    'candidate_count': len(candidates),
    'unique_category_patterns': unique_patterns,
    'unseen_positive_matches': len(positive_order),
    'top_score': float(scores[best]),
    'unseen_items_at_top_score': int(((scores == scores[best]) & ~seen).sum()),
}
display(pd.Series(geometry).to_frame('value'))


# Save the rank comparison and category-pattern counts for the interpretation.
tie_check.to_csv(OUT / 'tie_rule_comparison.csv', index=False)
(OUT / 'geometry.json').write_text(json.dumps(geometry, indent=2))


### Read the result

Saved-run finding: 1,765 candidates have only 62 unique label patterns; 564 unseen candidates have positive overlap and four share the highest score. The popularity tie rule puts Palais Garnier first; ID-only ordering puts Le Grand Rex first. The top ten lean toward shows. A broader menu would require an explicit separate list rule and evaluation.

## 8A. Extend the features as the business learns more

The 1,765 candidates in this run have only 62 distinct category combinations. The current representation therefore cannot distinguish many attractions, even when their actual experiences differ. This is a reason to investigate richer features, not evidence that every tied pair should receive a different score.

A business does not have to keep the same broad labels forever. It can gradually collect more detailed information and test whether it helps visitors choose.

| Information to acquire over time | Possible source and practical purpose |
|---|---|
| Finer topics: modern art, natural history, science | Attraction partners or a reviewed content catalogue; distinguish places inside the same broad category |
| Experience details: hands-on exhibits, guided activities, quiet viewing | Structured partner descriptions and editorial checks; describe what visitors actually do |
| Visitor’s stated interests for this trip | Optional preference questions; avoid assuming old likes describe today's intention |
| Which lists were shown and what happened next | Recommendation and response records; distinguish an unseen suggestion from one that was shown, and evaluate the resulting policy |

These are proposed collection directions, not available features or observed results in this run. Use a shared feature definition across attractions, retain unknown values rather than treating them as confirmed absence, and record when information was acquired or verified. Verify essential requirements such as access and opening hours separately; they should not be traded off against category similarity.

For an illustration, two museums currently share the same broad label. Finer attributes might distinguish an interactive science museum from a quiet modern-art gallery. Cosine can compare richer vectors, but feature construction, relative weights and missing-data handling must be defined first. Evaluate against the current category method before claiming an improvement.

Business decision: collect information that could change a visitor's choice, rather than adding fields merely to make the model larger.

## 9. Fill gaps honestly

Use positive category matches first; then fill with historical city popularity, always excluding reviewed places and duplicates. Every row records its source. A popularity filler can have a zero cosine: this means no shared category signal, not a dislike. Return fewer than ten if the eligible catalogue is too small. The broader approved 4-neighbour + 3-interest + 3-popular example remains a possible composition, not a quota; this notebook implements the interest-to-popularity part only.

In [ ]:
# Return up to the requested number of unseen places, with the score and real recommendation source kept separate.
def make_list(query, seen_mask=None, capacity=10):

    # Read the 20 interest weights; reject a wrong shape, missing values or negative weights.
    q = np.asarray(query, dtype=float)
    if q.shape != (20,) or not np.isfinite(q).all() or (q < 0).any():
        raise ValueError('Use 20 finite nonnegative category weights.')

    # Reject an invalid list length before slicing arrays.
    if isinstance(capacity, bool) or not isinstance(capacity, int) or capacity < 0:
        raise ValueError('Capacity must be a nonnegative integer.')

    # Build one reviewed/not-reviewed flag per candidate. With no supplied mask, nothing is excluded.
    excluded = (
        np.zeros(len(candidates), dtype=bool)
        if seen_mask is None
        else np.asarray(seen_mask, dtype=bool)
    )
    if excluded.shape != (len(candidates),):
        raise ValueError('Seen mask must align with candidates.')

    # A person with no usable interests gets no content matches; use zero scores so popularity can fill the list.
    norm = np.linalg.norm(q)
    local_scores = unit_categories[candidates] @ (q / norm) if norm else np.zeros(len(candidates))

    # Rank positive matches first and remove previously reviewed places before taking any rows.
    ranked = np.lexsort((ids, -support[candidates], -local_scores))
    matches = ranked[~excluded[ranked] & (local_scores[ranked] > 0)][:capacity]

    # Fill remaining positions with popular unseen places, skipping anything already chosen.
    pop = np.lexsort((ids, -support[candidates]))
    fillers = pop[~excluded[pop] & ~np.isin(pop, matches)][:max(0, capacity - len(matches))]

    # Keep the matching rows first, followed by any popular fillers.
    chosen = np.r_[matches, fillers]

    # Add names and both evidence fields so a reader can tell a category match from a popular filler.
    result = items.iloc[candidates[chosen]][['attractionID', 'nameAttraction', 'locationAttraction']].copy().reset_index(
        drop=True,
    )
    result['cosine'] = local_scores[chosen]
    result['positive_reviewers'] = support[candidates[chosen]]

    # Label every position honestly; a popular filler is not an individual-interest match.
    result['source'] = ['Own-interest category match'] * len(matches) + ['Destination popularity'] * len(fillers)
    result['fallback'] = [False] * len(matches) + [True] * len(fillers)
    return result


# Apply the rule to the real person and save the actual recommendation list.
final_list = make_list(profile, seen, K)
display(final_list)
final_list.to_csv(OUT / 'real_recommendations.csv', index=False)


# Save the profile, scores and row mappings so another process can check the calculation.
np.savez(
    OUT / 'score_reload.npz',
    profile=profile,
    candidates=candidates,
    scores=scores,
    seen=seen,
)
print(
    'This list helps narrow choices. Opening, access, price and today’s interests still require separate information.',
)


### Read the result

Saved-run finding: all ten entries for this person have positive category support, so none needs fallback. Do not imply that the real example demonstrates a mixed list. Section 11 constructs a clearly labelled mixed-support scenario.

## 10. How fragile is this interest picture?

A profile can be computed from one liked place, but that does not establish a stable preference. Remove each liked place in turn and recompute the top ten. Report shared items, not an accuracy claim. High overlap can also reflect broad identical labels and the popularity tie rule. For a very long history, inspect the first twenty removals in item-index order.

In [ ]:
# Keep the original recommended IDs so each changed list can be compared with the same baseline.
base_set = set(final_list.attractionID)
rows = []


# Remove one earlier liked place at a time; inspect at most the first 20 removals for a long history.
for position in range(min(len(liked), 20)):

    # Build interests from the remaining likes. If no likes remain, pass an empty-interest vector to fallback.
    remaining = np.delete(liked, position)
    q = unit_categories[remaining].mean(axis=0) if len(remaining) else np.zeros(20)

    # Generate a new list, then count how many of its items also appear in the original list.
    changed = make_list(q, seen, K)
    rows.append(
        {
            'removed_attraction': items.attractionID.iloc[liked[position]],
            'remaining_likes': len(remaining),
            'shared_top_items': len(base_set & set(changed.attractionID)),
            'base_list_size': len(base_set),
        },
    )


# Show how dependent this example list is on individual history records. This is not an accuracy score.
sensitivity = pd.DataFrame(rows)
display(sensitivity)
sensitivity.to_csv(OUT / 'leave_one_like_out.csv', index=False)
print(
    'A useful follow-up is to ask what this visitor wants today instead of assuming old likes are a complete preference.',
)


### Read the result

Saved-run finding: removing one of three likes retains only 4–6 of the original ten entries. This demonstrates dependence on sparse history for this person. It does not estimate average stability or prove poor ranking quality.

## 11. When to use it, ask a question, or stop

Use it when category labels are useful and the visitor has usable likes or explicitly selected interests. A new account can supply interests immediately. With no interests, use a clearly labelled starting list. A high cosine cannot establish wheelchair access, child suitability, affordability, opening hours or nearest distance. Those need separate verified information. We also do not subtract disliked categories here: disliking one museum does not prove disliking every museum.

In [ ]:
# Check that a single liked place can produce recommendations without a three-like requirement.
one_like = make_list(unit_categories[liked[0]], seen, K)


# With no usable interests, every returned place must be labelled as a popular filler.
no_history = make_list(np.zeros(20), seen, K)


# Invent a new visitor’s museum/park answers to demonstrate stated interests; these are not observed session inputs.
explicit = np.array([name in ['Museums', 'Nature...Parks'] for name in category_names], dtype=float)
stated = make_list(explicit, None, K)


# Deliberately leave only two candidates available and check that we return two instead of repeating items.
restricted = np.ones(len(candidates), dtype=bool)
restricted[:2] = False
short = make_list(np.zeros(20), restricted, K)
display(
    pd.DataFrame(
        {
            'scenario': [
                'One usable like',
                'No history or interests',
                'Stated museum/park interests (illustration)',
                'Only two eligible places remain',
            ],
            'returned': [len(one_like), len(no_history), len(stated), len(short)],
            'popularity_fillers': [int(x.fallback.sum()) for x in [one_like, no_history, stated, short]],
        },
    ),
)
display(no_history.head(3))
assert no_history.fallback.all() and len(short) == min(2, K)


# Save each boundary example with its own name so it cannot be confused with the real user list.
for name, frame in [
    ('one_like', one_like),
    ('no_history', no_history),
    ('stated_interests', stated),
    ('short_catalogue', short),
]:
    frame.to_csv(OUT / (name + '.csv'), index=False)


# Find unseen places with no category overlap for a controlled mixed-fallback example.
zero_order = np.lexsort((ids, -support[candidates]))


# Find unseen places with no category overlap for a controlled mixed-fallback example.
zero_order = zero_order[~seen[zero_order] & (scores[zero_order] <= 0)]
assert len(positive_order) >= 3 and len(zero_order) >= 7


# Leave exactly three real matches and seven zero-match places available. This shortage is imposed, not observed.
mixed_mask = np.ones(len(candidates), dtype=bool)
mixed_mask[np.r_[positive_order[:3], zero_order[:7]]] = False


# Check that the list contains three matches followed by seven clearly labelled popular fillers.
mixed = make_list(profile, mixed_mask, K)
display(mixed)
assert len(mixed) == 10 and int(mixed.fallback.sum()) == 7
mixed.to_csv(OUT / 'controlled_mixed_fallback.csv', index=False)


### Read the result

The controlled examples distinguish a one-like profile, no profile, stated interests, short supply and partial matching. In the deliberately constrained mixed example, three matches precede seven labelled popular fillers. It illustrates policy behaviour rather than an observed shortage for the real person.

## 12. Separate a matching score from evidence of usefulness

Three questions need different evidence: (1) Did the code follow the rules? (2) Does the list retrieve later recorded likes? (3) Did showing it improve the visitor’s choice? We can check the first now. The shared temporal evaluation is still under discussion, so this notebook does not create a new quality score for the real recommendations.

The invented example below explains Recall and NDCG. There are two later liked places, A and B. Both lists find both, so Recall@3 is one. Putting them first gives a higher NDCG. These are metric explanations, not real experiment results. Actual impact needs shown-list logs and a comparison design; a matching score alone cannot prove it.

In [ ]:
# Calculate Recall and NDCG for a tiny invented ranking whose positive outcomes we know completely.
def illustrated_metrics(ranking, positives, k=3):

    # Mark each displayed position 1 if it contains a known positive, otherwise 0.
    hits = np.array([item in positives for item in ranking[:k]], dtype=float)

    # Give an earlier hit more credit by dividing its value by log2(rank + 1).
    dcg = float((hits / np.log2(np.arange(len(hits)) + 2)).sum())

    # Compute the best possible discounted score for the available positives, so NDCG can be normalized.
    ideal = float((1 / np.log2(np.arange(min(k, len(positives))) + 2)).sum())

    # Return the share of positives retrieved and the discounted score divided by its ideal.
    return {'recall': float(hits.sum() / len(positives)), 'ndcg': dcg / ideal}


# Both invented lists find A and B, but one places them earlier. Show how the metrics treat that difference.
metric_example = pd.DataFrame(
    [
        {'list': 'A, B, C', **illustrated_metrics(['A', 'B', 'C'], {'A', 'B'})},
        {'list': 'C, A, B', **illustrated_metrics(['C', 'A', 'B'], {'A', 'B'})},
    ],
)
display(metric_example)


# Confirm the perfect-order case has NDCG one and both lists retrieve every positive.
assert np.isclose(metric_example.iloc[0].ndcg, 1) and (metric_example.recall == 1).all()
metric_example.to_csv(OUT / 'illustrated_metrics.csv', index=False)
print(
    'Business proposal: reduce search effort with explainable choices. Evidence of saved time or better choices is still to be collected.',
)


### Read the result

Both illustrative lists recover two of two positives, giving Recall@3 = 1. The first has NDCG@3 = 1; the second is approximately 0.693. Order can differ even when total hits do not. These figures are not real ranking-quality results for cosine.

## 13. Check the result and make the decision

Check score range, contribution arithmetic, ordering, unique results, reviewed-item exclusions, fallback flags and identical saved-score reload. Inspect the final names and reasons before adopting the list. A correct calculation supports an explainable starting shortlist; it does not settle whether category-only matching is sufficient.

Practitioner reflection: which missing detail would most change this visitor’s decision—today’s interest, access, cost, or a richer description? Choose the next data improvement from that decision rather than adding complexity automatically.

In [ ]:
# Check that the scores are real numbers in the expected range, not NaN or infinity.
assert np.isfinite(scores).all() and (scores >= -1e-12).all() and (scores <= 1 + 1e-12).all()


# Check that the positive-match scores are sorted from highest to lowest.
assert np.all(np.diff(scores[positive_order]) <= 1e-12)


# Reject repeated places, an overlong list or any place this person already reviewed.
assert final_list.attractionID.is_unique and len(final_list) <= K
assert not np.isin(final_list.attractionID, ids[seen]).any()


# Every row described as a category match needs positive overlap; fillers must follow those matches.
assert (final_list.loc[~final_list.fallback, 'cosine'] > 0).all()
assert final_list.fallback.tolist() == sorted(final_list.fallback.tolist())


# Check empty-capacity and all-reviewed cases: returning no places is valid.
assert make_list(profile, seen, 0).empty


# Check empty-capacity and all-reviewed cases: returning no places is valid.
assert make_list(profile, np.ones(len(candidates), dtype=bool), K).empty


# Reload the saved arrays and reproduce the scores instead of trusting the in-memory result.
reloaded = np.load(OUT / 'score_reload.npz')
assert np.array_equal(reloaded['scores'], unit_categories[reloaded['candidates']] @ reloaded['profile'])


# Save the observed findings with their scope, rather than presenting calculation checks as evidence of business impact.
summary = {
    'demo_user_index': DEMO_USER,
    'usable_likes': len(liked),
    'top_name': str(final_list.iloc[0].nameAttraction) if len(final_list) else None,
    'top_cosine': float(final_list.iloc[0].cosine) if len(final_list) else None,
    'own_interest_slots': int((~final_list.fallback).sum()),
    'popularity_slots': int(final_list.fallback.sum()),
    'candidate_count': len(candidates),
    'unique_category_patterns': unique_patterns,
    'leave_one_out_min_overlap': int(sensitivity.shared_top_items.min()),
    'leave_one_out_max_overlap': int(sensitivity.shared_top_items.max()),
    'checks': 'passed',
    'scope': 'category cosine walkthrough; no new real-data ranking-quality or business-impact evaluation',
}


# Write a short record of this run’s result and show it below.
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2))
display(pd.Series(summary).to_frame('result'))


### Read the result

Passing checks establishes calculation and policy consistency in this walkthrough. It does not establish a ranking improvement or business impact. The supported use is a transparent starting shortlist, with sparse-history and broad-category limitations visible.